# Cross-dataset Hi protocol-comparison tables

This notebook builds the cross-dataset tables for the Hi OOD-holdout versus random-shuffle validation analysis. It reads the per-fold artifacts produced by the nested-CV pipeline and aggregates performance, model-complexity and feature-importance results across the valid Hi datasets: DRD2-Hi, HIV-Hi and Sol-Hi.

KDR-Hi is intentionally excluded from this specific protocol comparison. Unlike DRD2-Hi, HIV-Hi and Sol-Hi, its outer training folds are restricted to 500 molecules and cannot be reconstructed as the union of the two non-test Hi subsets. Including KDR-Hi would therefore make the OOD-holdout and random-shuffle protocols train/refit on different data populations, breaking the matched comparison.

The analysis keeps the outer test fold fixed across protocols. Differences between OOD holdout and random shuffle therefore reflect the effect of the inner validation strategy on hyperparameter selection, selected model complexity and learned feature rankings, not a change in the external evaluation set.

Outputs are saved to:

```text
results/results_ood_vs_random_shuffle/hi/cross_dataset/

In [1]:
import json
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
try:
    PROJECT_ROOT = Path(__file__).resolve().parents[2]
except NameError:
    PROJECT_ROOT = Path.cwd()
    while (
        PROJECT_ROOT.name != "drug-discovery-lohi"
        and PROJECT_ROOT.parent != PROJECT_ROOT
    ):
        PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_ROOT = PROJECT_ROOT / "results" / "hi"

OUTPUT_DIR = (
    PROJECT_ROOT / "results" / "results_ood_vs_random_shuffle" / "hi" / "cross_dataset"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FOLDS = [1, 2, 3]

TOP_K_VALUES = [10, 20, 50, 100, 150, 200]

In [ ]:
_EXPERIMENT_TEMPLATES = [
    # Decision Tree
    (
        "Decision Tree",
        "DT",
        "ecfp4",
        "ECFP4",
        "OOD holdout",
        "dt_{dataset}_hi_inner_ood_holdout_ecfp4",
    ),
    (
        "Decision Tree",
        "DT",
        "ecfp4",
        "ECFP4",
        "Random shuffle",
        "dt_{dataset}_hi_random_shuffle_ecfp4",
    ),
    # Logistic Regression
    (
        "Logistic Regression",
        "LR",
        "ecfp4",
        "ECFP4",
        "OOD holdout",
        "lr_{dataset}_hi_inner_ood_holdout_ecfp4",
    ),
    (
        "Logistic Regression",
        "LR",
        "ecfp4",
        "ECFP4",
        "Random shuffle",
        "lr_{dataset}_hi_random_shuffle_ecfp4",
    ),
    # Linear SVM
    (
        "Linear SVM",
        "SVM",
        "ecfp4",
        "ECFP4",
        "OOD holdout",
        "svm_linear_{dataset}_hi_inner_ood_holdout_ecfp4",
    ),
    (
        "Linear SVM",
        "SVM",
        "ecfp4",
        "ECFP4",
        "Random shuffle",
        "svm_linear_{dataset}_hi_random_shuffle_ecfp4",
    ),
]

# KDR-Hi is intentionally excluded from this comparison because its
# outer training folds are restricted to 500 molecules and cannot be
# reconstructed as F_a ∪ F_b.
DATASETS = ["drd2", "hiv", "sol"]
EXCLUDED_DATASETS = ["kdr"]

DATASET_LABELS = {
    "drd2": "DRD2",
    "hiv": "HIV",
    "kdr": "KDR",
    "sol": "Sol",
}

ORDER_MODEL = {"Decision Tree": 0, "Logistic Regression": 1, "Linear SVM": 2}
ORDER_FP = {"ECFP4": 0, "MACCS": 1, "RDKit desc": 2}
ORDER_PROTOCOL = {"OOD holdout": 0, "Random shuffle": 1}
ORDER_DATASET = {"drd2": 0, "hiv": 1, "sol": 2, "kdr": 99}


def _build_registry() -> pd.DataFrame:
    """Build the full experiment registry across all datasets."""
    rows = []
    for dataset in DATASETS:
        dataset_dir = RESULTS_ROOT / dataset
        for (
            model,
            model_short,
            fp_type,
            fp_label,
            protocol,
            dir_template,
        ) in _EXPERIMENT_TEMPLATES:
            dir_name = dir_template.format(dataset=dataset)
            result_dir = dataset_dir / dir_name
            rows.append(
                {
                    "dataset": dataset,
                    "dataset_label": DATASET_LABELS[dataset],
                    "model": model,
                    "model_short": model_short,
                    "fp_type": fp_type,
                    "fingerprint": fp_label,
                    "protocol": protocol,
                    "dir_name": dir_name,
                    "result_dir": result_dir,
                    "exists": result_dir.exists(),
                }
            )
    df = pd.DataFrame(rows)
    if "kdr" in df["dataset"].unique():
        raise ValueError(
            "KDR should not be included in the OOD-vs-random Hi protocol tables. "
            "KDR-Hi has 500-molecule outer training folds and cannot be reconstructed "
            "as train_i = F_a ∪ F_b."
        )
    n_found = df["exists"].sum()
    n_total = len(df)
    print(f"Registry: {n_found}/{n_total} experiment directories found.")
    missing = df[~df["exists"]]
    if len(missing) > 0:
        for _, r in missing.iterrows():
            warnings.warn(f"Missing: {r['dataset']}/{r['dir_name']}")
    return df

In [4]:
def _read_json(path: Path) -> dict:
    with open(path, "r") as f:
        return json.load(f)


def _add_ordering_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if len(df) == 0:
        return df

    dataset_order = {
        "drd2": 0,
        "hiv": 1,
        "sol": 2,
        "kdr": 99,
    }

    model_order = {
        "Decision Tree": 0,
        "Logistic Regression": 1,
        "Linear SVM": 2,
        "DT": 0,
        "LR": 1,
        "SVM": 2,
    }

    fingerprint_order = {
        "ECFP4": 0,
        "ecfp4": 0,
        "MACCS": 1,
        "maccs": 1,
        "RDKit desc": 2,
        "rdkit_desc": 2,
    }

    protocol_order = {
        "OOD holdout": 0,
        "Random shuffle": 1,
        "kfold": 2,
        "K-fold": 2,
    }

    if "dataset" in df.columns:
        df["dataset_order"] = df["dataset"].map(dataset_order).fillna(99)
    else:
        df["dataset_order"] = 99

    if "model" in df.columns:
        df["model_order"] = df["model"].map(model_order).fillna(99)
    elif "model_short" in df.columns:
        df["model_order"] = df["model_short"].map(model_order).fillna(99)
    else:
        df["model_order"] = 99

    if "fingerprint" in df.columns:
        df["fingerprint_order"] = df["fingerprint"].map(fingerprint_order).fillna(99)
    else:
        df["fingerprint_order"] = 99

    if "protocol" in df.columns:
        df["protocol_order"] = df["protocol"].map(protocol_order).fillna(99)
    else:
        df["protocol_order"] = 99

    return df

## Protocol per-fold table

In [ ]:
def build_protocol_per_fold(registry: pd.DataFrame) -> pd.DataFrame:
    """Load params_fold_i.json for every experiment and build per-fold table."""
    rows = []
    for _, exp in registry[registry["exists"]].iterrows():
        for fold in FOLDS:
            params_path = exp["result_dir"] / f"params_fold_{fold}.json"
            if not params_path.exists():
                warnings.warn(f"Missing: {params_path}")
                continue

            data = _read_json(params_path)
            train_m = data.get("train_metrics", {})
            test_m = data.get("test_metrics", {})

            inner_raw = data.get("inner_selection_score", np.nan)

            try:
                inner = float(inner_raw)
            except (TypeError, ValueError):
                print(
                    f"WARNING: could not parse inner_selection_score={inner_raw!r} "
                    f"in {params_path}. Setting inner score to NaN."
                )
                inner = np.nan

            if not pd.isna(inner):

                if inner < -1e-9 or inner > 1.0 + 1e-9:
                    print(
                        f"WARNING: inner_selection_score={inner_raw!r} parsed as {inner:.6g} "
                        f"in {params_path}, outside expected PR-AUC range [0, 1]. "
                        "Setting inner score to NaN for gap-related analyses."
                    )
                    inner = np.nan
                else:
                    inner = min(max(inner, 0.0), 1.0)

            inner_train = data.get("inner_train_score", np.nan)
            train_pr = train_m.get("pr_auc", np.nan)
            test_pr = test_m.get("pr_auc", np.nan)
            inner_test_gap = (
                inner - test_pr
                if not pd.isna(inner) and not pd.isna(test_pr)
                else np.nan
            )  # inner validation PR-AUC - final outer test PR-AUC

            # granularity: dataset × model × fingerprint × protocol × fold
            # e.g. DRD2 | LR | ECFP4 | OOD holdout | fold 1 | inner_pr_auc | test_pr_auc | gap
            rows.append(
                {
                    "dataset": exp["dataset"],
                    "dataset_label": exp["dataset_label"],
                    "model": exp["model"],
                    "model_short": exp["model_short"],
                    "fingerprint": exp["fingerprint"],
                    "protocol": exp["protocol"],
                    "fold": fold,
                    "inner_pr_auc": inner,
                    "inner_train_pr_auc": inner_train,
                    "train_pr_auc": train_pr,
                    "test_pr_auc": test_pr,
                    "inner_test_gap": inner_test_gap,
                    "train_test_gap": train_pr
                    - test_pr,  # training PR-AUC - outer test PR-AUC
                }
            )

    df = pd.DataFrame(rows)
    df = _add_ordering_columns(df)
    sort_cols = [
        "dataset_order",
        "model_order",
        "fingerprint_order",
        "protocol_order",
        "fold",
    ]

    sort_cols = [c for c in sort_cols if c in df.columns]

    df = df.sort_values(sort_cols).reset_index(drop=True)

    return df

## Protocol summary (mean ± std across folds)

In [6]:
# granularity: dataset × model × fingerprint × protocol


def build_protocol_summary(per_fold: pd.DataFrame) -> pd.DataFrame:
    group_cols = [
        "dataset",
        "dataset_label",
        "model",
        "model_short",
        "fingerprint",
        "protocol",
    ]
    agg = per_fold.groupby(group_cols, as_index=False).agg(
        inner_mean=("inner_pr_auc", "mean"),
        inner_std=("inner_pr_auc", "std"),
        train_mean=("train_pr_auc", "mean"),
        train_std=("train_pr_auc", "std"),
        test_mean=("test_pr_auc", "mean"),
        test_std=("test_pr_auc", "std"),
        inner_test_gap_mean=("inner_test_gap", "mean"),  # inner_pr_auc - test_pr_auc
        inner_test_gap_std=("inner_test_gap", "std"),
        train_test_gap_mean=("train_test_gap", "mean"),  # train_pr_auc - test_pr_auc
        train_test_gap_std=("train_test_gap", "std"),
    )
    agg = _add_ordering_columns(agg)
    agg = agg.sort_values(
        ["dataset_order", "model_order", "fingerprint_order", "protocol_order"]
    ).reset_index(drop=True)
    return agg


# Is random shuffle more optimistic? --> Look at inner_mean and inner_test_gap_mean
# Does OOD holdout improve the final test? --> Look at test_mean for OOD and random.
# Does random select more overfitted models? Look at: train_test_gap_mean and complexity metrics

## Protocol delta table

In [7]:
# granularity: dataset × model × fingerprint


def build_protocol_delta(summary: pd.DataFrame) -> pd.DataFrame:
    """
    Compute deltas between Random shuffle and OOD holdout.

    Sign conventions:
      delta_inner_optimism = inner_random - inner_OOD   (positive = random inflated)
      delta_test_benefit   = test_OOD   - test_random   (positive = OOD better)
      delta_gap            = gap_random - gap_OOD       (positive = random more optimistic)
    """
    pivot = summary.pivot_table(
        index=["dataset", "dataset_label", "model", "model_short", "fingerprint"],
        columns="protocol",
        values=[
            "inner_mean",
            "test_mean",
            "inner_test_gap_mean",
            "train_test_gap_mean",
        ],
    )

    # e.g after pivot
    # dataset | model | fingerprint | inner_mean/OOD | inner_mean/Random | test_mean/OOD | test_mean/Random
    # DRD2    | LR    | ECFP4       | 0.75           | 0.91              | 0.76          | 0.77

    rows = []
    for idx in pivot.index:
        dataset, dataset_label, model, model_short, fingerprint = idx
        try:
            inner_ood = pivot.loc[idx, ("inner_mean", "OOD holdout")]
            inner_rnd = pivot.loc[idx, ("inner_mean", "Random shuffle")]
            test_ood = pivot.loc[idx, ("test_mean", "OOD holdout")]
            test_rnd = pivot.loc[idx, ("test_mean", "Random shuffle")]
            gap_ood = pivot.loc[idx, ("inner_test_gap_mean", "OOD holdout")]
            gap_rnd = pivot.loc[idx, ("inner_test_gap_mean", "Random shuffle")]
            train_gap_ood = pivot.loc[idx, ("train_test_gap_mean", "OOD holdout")]
            train_gap_rnd = pivot.loc[idx, ("train_test_gap_mean", "Random shuffle")]
        except KeyError:
            continue

        rows.append(
            {
                "dataset": dataset,
                "dataset_label": dataset_label,
                "model": model,
                "model_short": model_short,
                "fingerprint": fingerprint,
                "inner_ood": inner_ood,
                "inner_random": inner_rnd,
                "delta_inner_optimism": inner_rnd
                - inner_ood,  # if it's positive --> random more optimistic
                "test_ood": test_ood,
                "test_random": test_rnd,
                "delta_test_benefit": test_ood
                - test_rnd,  # if it's positive --> OOD holdout is better on the final test set
                "gap_ood": gap_ood,
                "gap_random": gap_rnd,
                "delta_gap": gap_rnd
                - gap_ood,  # gap_random = inner_random - test_random --> gap_ood = inner_ood - test_ood --> delta_gap = gap_random - gap_ood
                "train_gap_ood": train_gap_ood,
                "train_gap_random": train_gap_rnd,  # train_gap_random > train_gap_ood --> random shuffle is more complex
            }
        )

        # The delta_test_benefit with 3-fold cross-validation should be interpreted qualitatively and accompanied by fold-level deltas

        # delta_inner_optimism --> Does random validation yield a higher internal score than OOD validation?
        # delta_gap --> Does random validation overestimate the final test result more than OOD validation?

    df = pd.DataFrame(rows)
    df = _add_ordering_columns(df)
    df = df.sort_values(
        ["dataset_order", "model_order", "fingerprint_order"]
    ).reset_index(drop=True)
    return df

## Complexity tables

In [8]:
# Does random shuffle select more complex / overfitted models than OOD holdout?
# granularity: dataset × model × fingerprint × protocol × fold


def build_complexity_all(registry: pd.DataFrame) -> pd.DataFrame:
    """Load complexity_fold_i.json + params for all experiments."""
    rows = []
    for _, exp in registry[registry["exists"]].iterrows():
        for fold in FOLDS:
            params_path = exp["result_dir"] / f"params_fold_{fold}.json"
            complexity_path = exp["result_dir"] / f"complexity_fold_{fold}.json"

            if not params_path.exists() or not complexity_path.exists():
                continue

            params = _read_json(params_path)
            complexity = _read_json(complexity_path)

            train_m = params.get("train_metrics", {})
            test_m = params.get("test_metrics", {})

            inner = params.get("inner_selection_score", np.nan)
            train_pr = train_m.get("pr_auc", np.nan)
            test_pr = test_m.get("pr_auc", np.nan)

            row = {
                "dataset": exp["dataset"],
                "dataset_label": exp["dataset_label"],
                "model": exp["model"],
                "model_short": exp["model_short"],
                "fingerprint": exp["fingerprint"],
                "protocol": exp["protocol"],
                "fold": fold,
                "inner_pr_auc": inner,  # inner_pr_auc = internal model selection score
                "train_pr_auc": train_pr,  # train_pr_auc = PR-AUC on the full outer training set after refit
                "test_pr_auc": test_pr,  # test_pr_auc = PR-AUC on the outer test set
                "inner_test_gap": inner - test_pr,
                "train_test_gap": train_pr - test_pr,
            }
            # Add all complexity fields
            for key, value in complexity.items():
                row[key] = value

            rows.append(row)

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df
    df = _add_ordering_columns(df)
    df = df.sort_values(
        ["dataset_order", "model_order", "fingerprint_order", "protocol_order", "fold"]
    ).reset_index(drop=True)
    return df


def build_complexity_summary(complexity_all: pd.DataFrame) -> pd.DataFrame:
    """Aggregate complexity metrics across folds."""
    if len(complexity_all) == 0:
        return pd.DataFrame()

    indicator_cols = [
        "inner_pr_auc",
        "test_pr_auc",
        "inner_test_gap",
        "train_test_gap",
        "l2_norm",  # higher l2_norm → linear model with larger weights
        "l1_norm",
        "n_nonzero_coefficients",  # higher n_nonzero → more features actually used
        "sparsity",  # higher sparsity → more coefficients set to zero
        "approx_margin",  # lower approx_margin → narrower approximate margin, especially for linear SVMs
        "effective_depth",  # High effective_depth → deeper tree
        "n_nodes",  # Higher n_nodes → larger tree
        "n_leaves",  # Higher n_leaves → more decision regions
        "n_features_used",  # High n_features_used → uses more features for splitting
        "feature_min_depth_mean",
    ]

    # Thesis: Random validation is easier / more in-distribution, so it may favour more complex models,
    # which perform very well in validation but do not improve on the out-of-distribution test.

    available = [c for c in indicator_cols if c in complexity_all.columns]

    group_cols = [
        "dataset",
        "dataset_label",
        "model",
        "model_short",
        "fingerprint",
        "protocol",
    ]

    if not available:
        return pd.DataFrame(columns=group_cols)

    agg = complexity_all.groupby(group_cols, as_index=False).agg(
        {col: ["mean", "std"] for col in available}
    )

    # Flatten MultiIndex columns produced by aggregation
    flat_cols = []
    for col in agg.columns:
        if isinstance(col, tuple):
            parts = [str(c) for c in col if c]
            flat_cols.append("_".join(parts))
        else:
            flat_cols.append(col)

    agg.columns = flat_cols

    agg = _add_ordering_columns(agg)
    agg = agg.sort_values(
        ["dataset_order", "model_order", "fingerprint_order", "protocol_order"]
    ).reset_index(drop=True)

    return agg

## Feature importance tables

In [9]:
# Do OOD holdout and random shuffle cause the model to use the same molecular features, or different ones?
# Are the key features concentrated in a few bits or descriptors, or are they spread across many features?


def _unify_importance(
    fi: pd.DataFrame,
    model: str,
    dt_importance_mode: str = "permutation",
) -> pd.DataFrame:
    """
    Unify feature-importance rankings.

    Decision Tree:
      dt_importance_mode = "tree"
          main ranking = tree_importance
      dt_importance_mode = "permutation"
          main ranking = permutation_importance_mean, fallback tree_importance

    Logistic Regression / Linear SVM:
      main ranking = normalized_abs_importance, fallback abs_weight
    """
    fi = fi.copy()

    valid_importance = False

    if model == "Decision Tree":
        if dt_importance_mode == "tree":
            if "tree_importance" in fi.columns and fi["tree_importance"].notna().any():
                fi["importance_value"] = fi["tree_importance"]
                fi["importance_source"] = "tree_importance"

                if "rank_tree_importance" in fi.columns:
                    fi["importance_rank"] = fi["rank_tree_importance"]
                else:
                    fi["importance_rank"] = fi["importance_value"].rank(
                        ascending=False, method="first"
                    )
            else:
                fi["importance_value"] = np.nan
                fi["importance_rank"] = np.nan
                fi["importance_source"] = "none"

        elif dt_importance_mode == "permutation":
            if (
                "permutation_importance_mean" in fi.columns
                and fi["permutation_importance_mean"].notna().any()
            ):
                fi["importance_value"] = fi["permutation_importance_mean"]
                fi["importance_source"] = "permutation_importance_mean"

                if "permutation_importance_rank" in fi.columns:
                    fi["importance_rank"] = fi["permutation_importance_rank"]
                else:
                    fi["importance_rank"] = fi["importance_value"].rank(
                        ascending=False, method="first"
                    )

            elif "tree_importance" in fi.columns:
                fi["importance_value"] = fi["tree_importance"]
                fi["importance_source"] = "tree_importance"

                if "rank_tree_importance" in fi.columns:
                    fi["importance_rank"] = fi["rank_tree_importance"]
                else:
                    fi["importance_rank"] = fi["importance_value"].rank(
                        ascending=False, method="first"
                    )
            else:
                fi["importance_value"] = np.nan
                fi["importance_rank"] = np.nan
                fi["importance_source"] = "none"

        else:
            raise ValueError(
                "dt_importance_mode must be either 'tree' or 'permutation', "
                f"got {dt_importance_mode!r}"
            )

    else:
        if (
            "normalized_abs_importance" in fi.columns
            and fi["normalized_abs_importance"].notna().any()
        ):
            fi["importance_value"] = fi["normalized_abs_importance"]
            fi["importance_source"] = "normalized_abs_importance"
            valid_importance = True

        elif "abs_weight" in fi.columns and fi["abs_weight"].notna().any():
            fi["importance_value"] = fi["abs_weight"]
            fi["importance_source"] = "abs_weight"
            valid_importance = True

        else:
            fi["importance_value"] = np.nan
            fi["importance_source"] = "none"
            valid_importance = False

        if valid_importance and "rank_abs_weight" in fi.columns:
            fi["importance_rank"] = fi["rank_abs_weight"]
        elif valid_importance:
            fi["importance_rank"] = fi["importance_value"].rank(
                ascending=False, method="first"
            )
        else:
            fi["importance_rank"] = np.nan

    fi["importance_rank"] = fi["importance_rank"].astype("Int64")

    fi = fi.sort_values(
        ["importance_rank", "feature_idx"],
        ascending=[True, True],
        na_position="last",
    ).reset_index(drop=True)

    return fi


# granularity: dataset × model × fingerprint × protocol × fold × feature


def build_feature_importance_all(
    registry: pd.DataFrame,
    dt_importance_mode: str = "permutation",
) -> pd.DataFrame:
    """Load and unify feature importance CSVs for all experiments."""
    parts = []

    for _, exp in registry[registry["exists"]].iterrows():
        for fold in FOLDS:
            fi_path = exp["result_dir"] / f"feature_importance_fold_{fold}.csv"
            if not fi_path.exists():
                continue

            fi = pd.read_csv(fi_path)
            fi = _unify_importance(
                fi,
                exp["model"],
                dt_importance_mode=dt_importance_mode,
            )

            fi["dataset"] = exp["dataset"]
            fi["dataset_label"] = exp["dataset_label"]
            fi["model"] = exp["model"]
            fi["model_short"] = exp["model_short"]
            fi["fingerprint"] = exp["fingerprint"]
            fi["protocol"] = exp["protocol"]
            fi["fold"] = fold
            fi["dt_importance_mode"] = dt_importance_mode

            parts.append(fi)

    if not parts:
        return pd.DataFrame()

    return pd.concat(parts, ignore_index=True)


# for every dataset × model × fingerprint × protocol × fold it takes the first k features ordered by importance_rank


def build_feature_topk(fi_all: pd.DataFrame) -> pd.DataFrame:
    """
    Extract top-k positive-importance features per experiment × fold.

    We exclude zero or negative importance values before taking top-k.
    This avoids artificial overlap caused by tied zero-importance features
    ordered only by feature_idx.
    """
    if len(fi_all) == 0:
        return pd.DataFrame()

    group = ["dataset", "model", "fingerprint", "protocol", "fold"]

    df = fi_all.copy()
    df["importance_value"] = pd.to_numeric(df["importance_value"], errors="coerce")

    # Keep only features that actually carry positive importance.
    df = df[df["importance_value"] > 0].copy()

    parts = []

    for k in TOP_K_VALUES:
        topk = (
            df.sort_values(group + ["importance_rank", "feature_idx"])
            .groupby(group, as_index=False)
            .head(k)
            .copy()
        )

        topk["top_k"] = k
        parts.append(topk)

    if not parts:
        return pd.DataFrame()

    return pd.concat(parts, ignore_index=True)


# Do OOD holdout and random shuffle learn the same features? --> feature overlap

# If overlap is high --> the two protocols select similar features
# If overlap is low --> the validation protocol strongly influences which features are deemed important


def build_feature_overlap(fi_topk: pd.DataFrame) -> pd.DataFrame:
    """Compute top-k overlap between OOD and Random for matched experiments."""
    if len(fi_topk) == 0:
        return pd.DataFrame()

    rows = []
    combos = (
        fi_topk[["dataset", "model", "fingerprint"]]
        .drop_duplicates()
        .sort_values(["dataset", "model", "fingerprint"])
    )

    for _, combo in combos.iterrows():
        dataset = combo["dataset"]
        model = combo["model"]
        fp = combo["fingerprint"]

        for fold in FOLDS:
            for k in TOP_K_VALUES:
                sub = fi_topk[
                    (fi_topk["dataset"] == dataset)
                    & (fi_topk["model"] == model)
                    & (fi_topk["fingerprint"] == fp)
                    & (fi_topk["fold"] == fold)
                    & (fi_topk["top_k"] == k)
                ]

                # indices of the top-k features selected by the OOD model and the random model
                ood_feats = set(
                    sub.loc[
                        sub["protocol"] == "OOD holdout",
                        "feature_idx",
                    ].astype(int)
                )

                rnd_feats = set(
                    sub.loc[
                        sub["protocol"] == "Random shuffle",
                        "feature_idx",
                    ].astype(int)
                )

                if not ood_feats or not rnd_feats:
                    continue

                n_ood = len(ood_feats)
                n_random = len(rnd_feats)
                n_overlap = len(ood_feats & rnd_feats)
                n_union = len(ood_feats | rnd_feats)

                n_features_by_fp = {
                    "ECFP4": 2048,
                    "MACCS": 167,
                    "RDKit desc": 217,
                }

                n_features = n_features_by_fp.get(fp, np.nan)

                if n_ood == 0 or n_random == 0:
                    continue

                # Conservative top-k overlap.
                effective_k = min(k, max(n_ood, n_random))

                if effective_k == 0:
                    continue

                overlap_frac = n_overlap / effective_k
                jaccard_frac = n_overlap / n_union if n_union > 0 else np.nan

                if np.isfinite(n_features) and n_features > 0:
                    expected_overlap_count = (n_ood * n_random) / n_features
                    random_expected_percent = 100 * expected_overlap_count / effective_k
                else:
                    expected_overlap_count = np.nan
                    random_expected_percent = np.nan

                rows.append(
                    {
                        "dataset": dataset,
                        "model": model,
                        "fingerprint": fp,
                        "fold": fold,
                        "top_k": k,
                        "n_ood_features": n_ood,
                        "n_random_features": n_random,
                        "effective_k": effective_k,
                        "n_overlap": n_overlap,
                        "n_union": n_union,
                        "overlap_percent": round(100 * overlap_frac, 2),
                        "jaccard_percent": (
                            round(100 * jaccard_frac, 2)
                            if not np.isnan(jaccard_frac)
                            else np.nan
                        ),
                        "expected_overlap_count_random": (
                            round(expected_overlap_count, 4)
                            if not np.isnan(expected_overlap_count)
                            else np.nan
                        ),
                        "random_expected_percent": (
                            round(random_expected_percent, 2)
                            if not np.isnan(random_expected_percent)
                            else np.nan
                        ),
                    }
                )

    df = pd.DataFrame(rows)

    if len(df) > 0:
        df = _add_ordering_columns(df)
        df = df.sort_values(
            [
                "dataset_order",
                "model_order",
                "fingerprint_order",
                "fold",
                "top_k",
            ]
        ).reset_index(drop=True)

        duplicate_cols = ["dataset", "model", "fingerprint", "fold", "top_k"]
        n_duplicates = df.duplicated(subset=duplicate_cols).sum()

        if n_duplicates > 0:
            raise ValueError(
                f"Found {n_duplicates} duplicated feature-overlap rows. "
                "Each dataset/model/fingerprint/fold/top_k combination should appear once."
            )

    return df


# Is the model based on a few dominant features or on many distributed features? --> feature concentration

# If the top 20 capture a high proportion of the information --> the model is concentrated on a few bits/descriptors
# If the top 20 capture little information --> the model uses more distributed signals


def build_feature_concentration(fi_all: pd.DataFrame) -> pd.DataFrame:
    """
    Compute cumulative importance captured by top-k features.

    For permutation-based importance, negative values are clipped to zero.
    For tree_importance and coefficient-based rankings, importances are already
    non-negative after the chosen transformation.
    """
    if len(fi_all) == 0:
        return pd.DataFrame()

    group = [
        "dataset",
        "model",
        "model_short",
        "fingerprint",
        "protocol",
        "fold",
    ]

    rows = []

    for keys, sub in fi_all.groupby(group):
        dataset, model, model_short, fp, protocol, fold = keys

        raw_imp = sub["importance_value"].fillna(0).astype(float).values
        clipped_imp = np.clip(raw_imp, a_min=0.0, a_max=None)

        sorted_imp = np.sort(clipped_imp)[::-1]
        total = float(sorted_imp.sum())

        row = {
            "dataset": dataset,
            "model": model,
            "model_short": model_short,
            "fingerprint": fp,
            "protocol": protocol,
            "fold": fold,
            "n_features": int(len(raw_imp)),
            "n_nonzero_raw_importance": int(np.sum(raw_imp != 0)),
            "n_positive_importance": int(np.sum(clipped_imp > 0)),
            "n_negative_importance": int(
                np.sum(raw_imp < 0)
            ),  # how many features have a negative permutation importance
            "total_importance_raw": float(np.sum(raw_imp)),
            "total_importance_positive": total,
        }

        for k in TOP_K_VALUES:
            # calculate the cumulative importance of the top-k features
            cum = float(np.sum(sorted_imp[:k]))
            row[f"cumulative_top_{k}"] = cum

            # e.g. fraction_top_20 = top 20 importance / total positive importance
            # fraction_top_20 = 0.80 --> the top 20 features account for 80% of the total positive importance
            if total > 0:
                row[f"fraction_top_{k}"] = round(cum / total, 6)
            else:
                row[f"fraction_top_{k}"] = 0.0

        rows.append(row)

    df = pd.DataFrame(rows)

    if len(df) == 0:
        return df

    df = _add_ordering_columns(df)

    protocol_order = {
        "OOD holdout": 0,
        "Random shuffle": 1,
        "kfold": 2,
        "K-fold": 2,
    }

    if "protocol" in df.columns:
        df["protocol_order"] = df["protocol"].map(protocol_order).fillna(99)
    else:
        df["protocol_order"] = 99

    if "fold" not in df.columns:
        df["fold"] = 99

    sort_cols = [
        "dataset_order",
        "model_order",
        "fingerprint_order",
        "protocol_order",
        "fold",
    ]

    sort_cols = [c for c in sort_cols if c in df.columns]

    df = df.sort_values(sort_cols).reset_index(drop=True)

    return df

In [10]:
def main():
    print(f"Project root: {PROJECT_ROOT}")
    print(f"Results root: {RESULTS_ROOT}")
    print(f"Output dir:   {OUTPUT_DIR}\n")

    registry = _build_registry()

    print("\n--- Building protocol tables ---")
    per_fold = build_protocol_per_fold(registry)
    per_fold.to_csv(OUTPUT_DIR / "cross_dataset_protocol_per_fold.csv", index=False)
    print(f"  Saved: cross_dataset_protocol_per_fold.csv  ({len(per_fold)} rows)")

    summary = build_protocol_summary(per_fold)
    summary.to_csv(OUTPUT_DIR / "cross_dataset_protocol_summary.csv", index=False)
    print(f"  Saved: cross_dataset_protocol_summary.csv  ({len(summary)} rows)")

    delta = build_protocol_delta(summary)
    delta.to_csv(OUTPUT_DIR / "cross_dataset_protocol_delta.csv", index=False)
    print(f"  Saved: cross_dataset_protocol_delta.csv  ({len(delta)} rows)")

    print("\n--- Building complexity tables ---")
    complexity_all = build_complexity_all(registry)
    complexity_all.to_csv(OUTPUT_DIR / "cross_dataset_complexity_all.csv", index=False)
    print(f"  Saved: cross_dataset_complexity_all.csv  ({len(complexity_all)} rows)")

    complexity_summary = build_complexity_summary(complexity_all)
    complexity_summary.to_csv(
        OUTPUT_DIR / "cross_dataset_complexity_summary.csv", index=False
    )
    print(
        f"  Saved: cross_dataset_complexity_summary.csv  ({len(complexity_summary)} rows)"
    )

    print("\n--- Building feature-importance tables ---")

    feature_outputs = {}

    for dt_mode in ["tree", "permutation"]:
        print(f"\n  Importance mode: {dt_mode}")

        fi_all = build_feature_importance_all(
            registry,
            dt_importance_mode=dt_mode,
        )

        fi_all_path = OUTPUT_DIR / f"cross_dataset_feature_importance_all_{dt_mode}.csv"
        fi_all.to_csv(fi_all_path, index=False)
        print(f"  Saved: {fi_all_path.name}  ({len(fi_all)} rows)")

        fi_topk = build_feature_topk(fi_all)

        fi_topk_path = OUTPUT_DIR / f"cross_dataset_feature_topk_{dt_mode}.csv"
        fi_topk.to_csv(fi_topk_path, index=False)
        print(f"  Saved: {fi_topk_path.name}  ({len(fi_topk)} rows)")

        overlap = build_feature_overlap(fi_topk)

        if len(overlap) > 0:
            duplicate_cols = ["dataset", "model", "fingerprint", "fold", "top_k"]
            n_duplicates = overlap.duplicated(subset=duplicate_cols).sum()

            if n_duplicates > 0:
                raise ValueError(
                    f"Feature-overlap table contains {n_duplicates} duplicated rows. "
                    "This would bias fold-level standard deviations in the plots."
                )

        overlap_path = OUTPUT_DIR / f"cross_dataset_feature_overlap_{dt_mode}.csv"
        overlap.to_csv(overlap_path, index=False)
        print(f"  Saved: {overlap_path.name}  ({len(overlap)} rows)")

        concentration = build_feature_concentration(fi_all)

        concentration_path = (
            OUTPUT_DIR / f"cross_dataset_feature_concentration_{dt_mode}.csv"
        )
        concentration.to_csv(concentration_path, index=False)
        print(f"  Saved: {concentration_path.name}  ({len(concentration)} rows)")

        feature_outputs[dt_mode] = {
            "fi_all": fi_all,
            "fi_topk": fi_topk,
            "overlap": overlap,
            "concentration": concentration,
        }

    # Main diagnostic version.
    MAIN_IMPORTANCE_MODE = "tree"

    fi_all = feature_outputs[MAIN_IMPORTANCE_MODE]["fi_all"]
    fi_topk = feature_outputs[MAIN_IMPORTANCE_MODE]["fi_topk"]
    overlap = feature_outputs[MAIN_IMPORTANCE_MODE]["overlap"]
    concentration = feature_outputs[MAIN_IMPORTANCE_MODE]["concentration"]

    print(
        f"\n--- Saving unsuffixed main feature tables: {MAIN_IMPORTANCE_MODE} mode ---"
    )

    fi_all.to_csv(
        OUTPUT_DIR / "cross_dataset_feature_importance_all.csv",
        index=False,
    )
    print(f"  Saved: cross_dataset_feature_importance_all.csv  ({len(fi_all)} rows)")

    fi_topk.to_csv(
        OUTPUT_DIR / "cross_dataset_feature_topk.csv",
        index=False,
    )
    print(f"  Saved: cross_dataset_feature_topk.csv  ({len(fi_topk)} rows)")

    overlap.to_csv(
        OUTPUT_DIR / "cross_dataset_feature_overlap.csv",
        index=False,
    )
    print(f"  Saved: cross_dataset_feature_overlap.csv  ({len(overlap)} rows)")

    concentration.to_csv(
        OUTPUT_DIR / "cross_dataset_feature_concentration.csv",
        index=False,
    )
    print(
        f"  Saved: cross_dataset_feature_concentration.csv  ({len(concentration)} rows)"
    )

    print(f"\nDone. All tables saved to:\n  {OUTPUT_DIR}")

    return {
        "registry": registry,
        "per_fold": per_fold,
        "summary": summary,
        "delta": delta,
        "complexity_all": complexity_all,
        "complexity_summary": complexity_summary,
        # Main diagnostic version, tree-based for DT
        "fi_all": fi_all,
        "fi_topk": fi_topk,
        "overlap": overlap,
        "concentration": concentration,
        # Explicit parallel versions
        "feature_outputs": feature_outputs,
        "fi_all_tree": feature_outputs["tree"]["fi_all"],
        "fi_topk_tree": feature_outputs["tree"]["fi_topk"],
        "overlap_tree": feature_outputs["tree"]["overlap"],
        "concentration_tree": feature_outputs["tree"]["concentration"],
        "fi_all_permutation": feature_outputs["permutation"]["fi_all"],
        "fi_topk_permutation": feature_outputs["permutation"]["fi_topk"],
        "overlap_permutation": feature_outputs["permutation"]["overlap"],
        "concentration_permutation": feature_outputs["permutation"]["concentration"],
    }

In [11]:
outputs = main()

Project root: /home/f.capria/drug-discovery-lohi
Results root: /home/f.capria/drug-discovery-lohi/results/hi
Output dir:   /home/f.capria/drug-discovery-lohi/results/results_ood_vs_random_shuffle/hi/cross_dataset

Registry: 34/42 experiment directories found.

--- Building protocol tables ---
  Saved: cross_dataset_protocol_per_fold.csv  (102 rows)
  Saved: cross_dataset_protocol_summary.csv  (34 rows)
  Saved: cross_dataset_protocol_delta.csv  (17 rows)

--- Building complexity tables ---
  Saved: cross_dataset_complexity_all.csv  (102 rows)
  Saved: cross_dataset_complexity_summary.csv  (34 rows)

--- Building feature-importance tables ---

  Importance mode: tree


/tmp/ipykernel_348138/712541392.py:180: UserWarning: Missing: hiv/dt_hiv_hi_inner_ood_holdout_maccs
  warnings.warn(f"Missing: {r['dataset']}/{r['dir_name']}")
/tmp/ipykernel_348138/712541392.py:180: UserWarning: Missing: hiv/dt_hiv_hi_random_shuffle_maccs
  warnings.warn(f"Missing: {r['dataset']}/{r['dir_name']}")
/tmp/ipykernel_348138/712541392.py:180: UserWarning: Missing: hiv/lr_hiv_hi_inner_ood_holdout_maccs
  warnings.warn(f"Missing: {r['dataset']}/{r['dir_name']}")
/tmp/ipykernel_348138/712541392.py:180: UserWarning: Missing: hiv/lr_hiv_hi_random_shuffle_maccs
  warnings.warn(f"Missing: {r['dataset']}/{r['dir_name']}")
/tmp/ipykernel_348138/712541392.py:180: UserWarning: Missing: hiv/lr_hiv_hi_inner_ood_holdout_rdkit_desc
  warnings.warn(f"Missing: {r['dataset']}/{r['dir_name']}")
/tmp/ipykernel_348138/712541392.py:180: UserWarning: Missing: hiv/lr_hiv_hi_random_shuffle_rdkit_desc
  warnings.warn(f"Missing: {r['dataset']}/{r['dir_name']}")
/tmp/ipykernel_348138/712541392.py:180:

  Saved: cross_dataset_feature_importance_all_tree.csv  (119208 rows)
  Saved: cross_dataset_feature_topk_tree.csv  (43018 rows)
  Saved: cross_dataset_feature_overlap_tree.csv  (306 rows)
  Saved: cross_dataset_feature_concentration_tree.csv  (102 rows)

  Importance mode: permutation
  Saved: cross_dataset_feature_importance_all_permutation.csv  (119208 rows)
  Saved: cross_dataset_feature_topk_permutation.csv  (39768 rows)
  Saved: cross_dataset_feature_overlap_permutation.csv  (306 rows)
  Saved: cross_dataset_feature_concentration_permutation.csv  (102 rows)

--- Saving unsuffixed main feature tables: tree mode ---
  Saved: cross_dataset_feature_importance_all.csv  (119208 rows)
  Saved: cross_dataset_feature_topk.csv  (43018 rows)
  Saved: cross_dataset_feature_overlap.csv  (306 rows)
  Saved: cross_dataset_feature_concentration.csv  (102 rows)

Done. All tables saved to:
  /home/f.capria/drug-discovery-lohi/results/results_ood_vs_random_shuffle/hi/cross_dataset


In [12]:
# Post-build consistency checks

# ---------------------------------------------------------------------
# 1. KDR exclusion checks
# ---------------------------------------------------------------------

base_output_names = [
    "per_fold",
    "summary",
    "delta",
    "complexity_all",
    "complexity_summary",
    "fi_all",
    "fi_topk",
    "overlap",
    "concentration",
]

for name in base_output_names:
    if name not in outputs:
        continue

    df = outputs[name]

    if isinstance(df, pd.DataFrame) and "dataset" in df.columns:
        assert "kdr" not in set(df["dataset"]), f"KDR found in {name}"

# Also check explicit tree/permutation feature outputs, if present.
for mode in ["tree", "permutation"]:
    for suffix_name in ["fi_all", "fi_topk", "overlap", "concentration"]:
        key = f"{suffix_name}_{mode}"

        if key not in outputs:
            continue

        df = outputs[key]

        if isinstance(df, pd.DataFrame) and "dataset" in df.columns:
            assert "kdr" not in set(df["dataset"]), f"KDR found in {key}"

print("OK: KDR excluded from all OOD-vs-random cross-dataset tables.")


# ---------------------------------------------------------------------
# 2. Feature-overlap duplicate checks
# ---------------------------------------------------------------------


def check_overlap_duplicates(overlap_df: pd.DataFrame, name: str) -> None:
    if overlap_df is None or len(overlap_df) == 0:
        print(f"WARNING: {name} is empty, duplicate check skipped.")
        return

    duplicate_cols = ["dataset", "model", "fingerprint", "fold", "top_k"]

    missing_cols = [c for c in duplicate_cols if c not in overlap_df.columns]
    assert (
        not missing_cols
    ), f"Missing duplicate-check columns in {name}: {missing_cols}"

    n_duplicates = overlap_df.duplicated(subset=duplicate_cols).sum()

    assert n_duplicates == 0, (
        f"Duplicated rows found in {name} for " "dataset/model/fingerprint/fold/top_k."
    )

    print(f"OK: {name} has no duplicated dataset/model/fingerprint/fold/top_k rows.")


# Main / unsuffixed overlap table.
check_overlap_duplicates(outputs["overlap"], "main overlap")

# Explicit tree/permutation overlap tables, if present.
if "overlap_tree" in outputs:
    check_overlap_duplicates(outputs["overlap_tree"], "tree overlap")

if "overlap_permutation" in outputs:
    check_overlap_duplicates(outputs["overlap_permutation"], "permutation overlap")


# ---------------------------------------------------------------------
# 3. Feature-importance source checks
# ---------------------------------------------------------------------


def check_feature_importance_sources(
    fi_all: pd.DataFrame,
    mode_name: str,
    expected_dt_source: str,
) -> pd.DataFrame:
    """
    Check that the unified importance_source is consistent.

    expected_dt_source:
        "tree_importance" for tree mode
        "permutation_importance_mean" for permutation mode
    """
    if fi_all is None or len(fi_all) == 0:
        print(f"WARNING: {mode_name}: fi_all is empty, source checks skipped.")
        return pd.DataFrame()

    required_cols = [
        "dataset",
        "model",
        "fingerprint",
        "protocol",
        "fold",
        "importance_source",
    ]

    missing_cols = [c for c in required_cols if c not in fi_all.columns]
    assert (
        not missing_cols
    ), f"{mode_name}: missing required columns in fi_all: {missing_cols}"

    group_cols = ["dataset", "model", "fingerprint", "protocol", "fold"]

    importance_source_summary = (
        fi_all.groupby(group_cols + ["importance_source"], as_index=False)
        .size()
        .rename(columns={"size": "n_features"})
        .sort_values(group_cols + ["importance_source"])
        .reset_index(drop=True)
    )

    print(f"\n--- Feature-importance source summary: {mode_name} ---")
    display(importance_source_summary)

    # Decision Tree source check.
    dt_bad = importance_source_summary[
        (importance_source_summary["model"] == "Decision Tree")
        & (importance_source_summary["importance_source"] != expected_dt_source)
    ]

    assert len(dt_bad) == 0, (
        f"{mode_name}: Some Decision Tree feature rankings did not use "
        f"{expected_dt_source}. Unexpected source detected:\n"
        f"{dt_bad.to_string(index=False)}"
    )

    # Logistic Regression source check.
    lr_allowed_sources = {"normalized_abs_importance", "abs_weight"}

    lr_bad = importance_source_summary[
        (importance_source_summary["model"] == "Logistic Regression")
        & (~importance_source_summary["importance_source"].isin(lr_allowed_sources))
    ]

    assert len(lr_bad) == 0, (
        f"{mode_name}: Some Logistic Regression feature rankings used an unexpected source:\n"
        f"{lr_bad.to_string(index=False)}"
    )

    # Linear SVM source check.
    svm_allowed_sources = {"normalized_abs_importance", "abs_weight"}

    svm_bad = importance_source_summary[
        (importance_source_summary["model"] == "Linear SVM")
        & (~importance_source_summary["importance_source"].isin(svm_allowed_sources))
    ]

    assert len(svm_bad) == 0, (
        f"{mode_name}: Some Linear SVM feature rankings used an unexpected source:\n"
        f"{svm_bad.to_string(index=False)}"
    )

    print(
        f"OK: {mode_name} feature-importance sources are consistent "
        f"(DT = {expected_dt_source}; LR/SVM = coefficient-based importance)."
    )

    return importance_source_summary


# Main / unsuffixed output should now be tree-based.
main_importance_source_summary = check_feature_importance_sources(
    outputs["fi_all"],
    mode_name="main",
    expected_dt_source="tree_importance",
)

# Explicit tree mode.
if "fi_all_tree" in outputs:
    tree_importance_source_summary = check_feature_importance_sources(
        outputs["fi_all_tree"],
        mode_name="tree",
        expected_dt_source="tree_importance",
    )

# Explicit permutation mode.
if "fi_all_permutation" in outputs:
    permutation_importance_source_summary = check_feature_importance_sources(
        outputs["fi_all_permutation"],
        mode_name="permutation",
        expected_dt_source="permutation_importance_mean",
    )

OK: KDR excluded from all OOD-vs-random cross-dataset tables.
OK: main overlap has no duplicated dataset/model/fingerprint/fold/top_k rows.
OK: tree overlap has no duplicated dataset/model/fingerprint/fold/top_k rows.
OK: permutation overlap has no duplicated dataset/model/fingerprint/fold/top_k rows.

--- Feature-importance source summary: main ---


,dataset,model,fingerprint,protocol,fold,importance_source,n_features
0,drd2,Decision Tree,ECFP4,OOD holdout,1,tree_importance,2048
1,drd2,Decision Tree,ECFP4,OOD holdout,2,tree_importance,2048
2,drd2,Decision Tree,ECFP4,OOD holdout,3,tree_importance,2048
3,drd2,Decision Tree,ECFP4,Random shuffle,1,tree_importance,2048
4,drd2,Decision Tree,ECFP4,Random shuffle,2,tree_importance,2048
...,...,...,...,...,...,...,...
97,sol,Logistic Regression,RDKit desc,OOD holdout,2,normalized_abs_importance,217
98,sol,Logistic Regression,RDKit desc,OOD holdout,3,normalized_abs_importance,217
99,sol,Logistic Regression,RDKit desc,Random shuffle,1,normalized_abs_importance,217
100,sol,Logistic Regression,RDKit desc,Random shuffle,2,normalized_abs_importance,217


OK: main feature-importance sources are consistent (DT = tree_importance; LR/SVM = coefficient-based importance).

--- Feature-importance source summary: tree ---


,dataset,model,fingerprint,protocol,fold,importance_source,n_features
0,drd2,Decision Tree,ECFP4,OOD holdout,1,tree_importance,2048
1,drd2,Decision Tree,ECFP4,OOD holdout,2,tree_importance,2048
2,drd2,Decision Tree,ECFP4,OOD holdout,3,tree_importance,2048
3,drd2,Decision Tree,ECFP4,Random shuffle,1,tree_importance,2048
4,drd2,Decision Tree,ECFP4,Random shuffle,2,tree_importance,2048
...,...,...,...,...,...,...,...
97,sol,Logistic Regression,RDKit desc,OOD holdout,2,normalized_abs_importance,217
98,sol,Logistic Regression,RDKit desc,OOD holdout,3,normalized_abs_importance,217
99,sol,Logistic Regression,RDKit desc,Random shuffle,1,normalized_abs_importance,217
100,sol,Logistic Regression,RDKit desc,Random shuffle,2,normalized_abs_importance,217


OK: tree feature-importance sources are consistent (DT = tree_importance; LR/SVM = coefficient-based importance).

--- Feature-importance source summary: permutation ---


,dataset,model,fingerprint,protocol,fold,importance_source,n_features
0,drd2,Decision Tree,ECFP4,OOD holdout,1,permutation_importance_mean,2048
1,drd2,Decision Tree,ECFP4,OOD holdout,2,permutation_importance_mean,2048
2,drd2,Decision Tree,ECFP4,OOD holdout,3,permutation_importance_mean,2048
3,drd2,Decision Tree,ECFP4,Random shuffle,1,permutation_importance_mean,2048
4,drd2,Decision Tree,ECFP4,Random shuffle,2,permutation_importance_mean,2048
...,...,...,...,...,...,...,...
97,sol,Logistic Regression,RDKit desc,OOD holdout,2,normalized_abs_importance,217
98,sol,Logistic Regression,RDKit desc,OOD holdout,3,normalized_abs_importance,217
99,sol,Logistic Regression,RDKit desc,Random shuffle,1,normalized_abs_importance,217
100,sol,Logistic Regression,RDKit desc,Random shuffle,2,normalized_abs_importance,217


OK: permutation feature-importance sources are consistent (DT = permutation_importance_mean; LR/SVM = coefficient-based importance).


In [13]:
outputs["summary"]
outputs["delta"]
outputs["fi_all"]

,feature_idx,feature_name,tree_importance,minimum_depth,used_in_tree,normalized_tree_importance,rank_tree_importance,permutation_importance_mean,permutation_importance_std,permutation_scoring,...,model_short,fingerprint,protocol,fold,dt_importance_mode,raw_weight,abs_weight,normalized_abs_importance,direction,rank_abs_weight
0,2000,ecfp4_bit_2000,0.095092,0.0,True,0.095092,1.0,0.007348,0.005241,average_precision,...,DT,ECFP4,OOD holdout,1,tree,NaN,NaN,NaN,NaN,NaN
1,1145,ecfp4_bit_1145,0.051899,5.0,True,0.051899,2.0,0.034229,0.004964,average_precision,...,DT,ECFP4,OOD holdout,1,tree,NaN,NaN,NaN,NaN,NaN
2,1503,ecfp4_bit_1503,0.039868,7.0,True,0.039868,3.0,-0.002270,0.001541,average_precision,...,DT,ECFP4,OOD holdout,1,tree,NaN,NaN,NaN,NaN,NaN
3,562,ecfp4_bit_562,0.032088,2.0,True,0.032088,4.0,-0.013747,0.003835,average_precision,...,DT,ECFP4,OOD holdout,1,tree,NaN,NaN,NaN,NaN,NaN
4,268,ecfp4_bit_268,0.031189,6.0,True,0.031189,5.0,0.003997,0.001195,average_precision,...,DT,ECFP4,OOD holdout,1,tree,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119203,9,maccs_bit_9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,SVM,MACCS,Random shuffle,3,tree,0.0,0.0,0.0,zero,163.0
119204,10,maccs_bit_10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,SVM,MACCS,Random shuffle,3,tree,0.0,0.0,0.0,zero,164.0
119205,12,maccs_bit_12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,SVM,MACCS,Random shuffle,3,tree,0.0,0.0,0.0,zero,165.0
119206,13,maccs_bit_13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,SVM,MACCS,Random shuffle,3,tree,0.0,0.0,0.0,zero,166.0
